In [1]:
import numpy as np
import pandas as pd
from itertools import combinations

FEATURES = ["A", "B", "C", "D", "E"]

def make_base_block(reps=25):
    """One timestep of data: all 4 combinations of the two latent bits (u, v),
    each repeated `reps` times -> 100 rows.

        A = u        B = u                  (the "u-channel")
        C = v        D = v      E = 1 - v   (the "v-channel")
        Y = u XOR v
    """
    u = np.array([1, 1, 0, 0])
    v = np.array([1, 0, 1, 0])
    block = pd.DataFrame({
        "A": u,
        "B": u,
        "C": v,
        "D": v,
        "E": 1 - v,
        "Y": u ^ v,
    })
    return pd.concat([block] * reps, ignore_index=True).astype(np.int8)


def determines(df, cols):
    """Gavin's `cols -> Y` notation, computed rather than asserted.
    Groups rows by the values of `cols` and checks Y never disagrees
    within a group. True = this feature set pins Y down exactly."""
    return df.groupby(list(cols), observed=True)["Y"].nunique().max() == 1


base = make_base_block()

# --- ground truth of the construction ---
assert base.shape == (100, 6)
assert (base["A"] == base["B"]).all()              # B duplicates A
assert (base["C"] == base["D"]).all()              # D duplicates C
assert (base["E"] == 1 - base["C"]).all()          # E complements C
assert (base["Y"] == (base["A"] ^ base["C"])).all()  # Y = A XOR C  (NOT A XOR B)
assert base["Y"].value_counts().to_dict() == {0: 50, 1: 50}

# --- ground truth of the reliance structure ---
singletons = [f for f in FEATURES if determines(base, [f])]
pairs      = [p for p in combinations(FEATURES, 2) if determines(base, p)]
assert singletons == [], "no single feature should determine Y"
assert set(pairs) == {("A","C"),("A","D"),("A","E"),
                      ("B","C"),("B","D"),("B","E")}

print("singletons -> Y:", singletons or "none (correct)")
print("pairs -> Y:     ", pairs)
print()
print(base.head(4).to_string(index=False))
print("\nshape:", base.shape)

singletons -> Y: none (correct)
pairs -> Y:      [('A', 'C'), ('A', 'D'), ('A', 'E'), ('B', 'C'), ('B', 'D'), ('B', 'E')]

 A  B  C  D  E  Y
 1  1  1  1  0  0
 1  1  0  0  1  1
 0  0  1  1  0  1
 0  0  0  0  1  0

shape: (100, 6)


In [2]:
def make_schedule(n_rows, n_steps, rng):
    """Without-replacement row scheduler: splits every row index into
    `n_steps` disjoint groups, so after the final step the whole table
    has been touched exactly once and never twice."""
    assert n_rows % n_steps == 0, "n_rows must divide evenly into n_steps"
    return rng.permutation(n_rows).reshape(n_steps, n_rows // n_steps)


def swap_cells(df, rows, col_x, col_y):
    """P1 primitive (A <-> D): exchange two columns' values on `rows` only.
    Two-way -- both columns change. Mutates df in place."""
    x = df.loc[rows, col_x].to_numpy(copy=True)
    y = df.loc[rows, col_y].to_numpy(copy=True)
    df.loc[rows, col_x] = y
    df.loc[rows, col_y] = x


def copy_cells(df, rows, src, dst):
    """P2 primitive (B <- E): overwrite dst with src on `rows` only.
    One-way -- src is untouched. Mutates df in place."""
    df.loc[rows, dst] = df.loc[rows, src].to_numpy(copy=True)


# --- verification on a single P1 step ---
assert isinstance(base.index, pd.RangeIndex)   # row labels == positions

rng   = np.random.default_rng(0)
sched = make_schedule(len(base), n_steps=10, rng=rng)
assert sched.shape == (10, 10)
assert sorted(sched.ravel().tolist()) == list(range(100)), "must cover each row once"

df = base.copy()
rows = sched[0]
swap_cells(df, rows, "A", "D")

# labels are never touched -- this is feature drift, not label noise
assert (df["Y"] == base["Y"]).all()
# columns not named in the swap are untouched
for c in ["B", "C", "E"]:
    assert (df[c] == base[c]).all()
# on the swapped rows, A now carries the v-channel and D carries the u-channel
r = np.sort(rows)
assert (df.loc[r, "A"] == base.loc[r, "C"]).all()
assert (df.loc[r, "D"] == base.loc[r, "A"]).all()

altered = df.index[(df["A"] != base["A"]) | (df["D"] != base["D"])]
print(f"rows scheduled: {len(rows)}   rows whose values actually changed: {len(altered)}")
print("A^C -> Y ?", determines(df, ["A", "C"]))
print("B^C -> Y ?", determines(df, ["B", "C"]))
print("D^C -> Y ?", determines(df, ["D", "C"]))
print("dtypes:", set(df.dtypes.astype(str)))

rows scheduled: 10   rows whose values actually changed: 5
A^C -> Y ? False
B^C -> Y ? True
D^C -> Y ? False
dtypes: {'int8'}


In [3]:
# Phase spec: (name, kind, columns). "swap" is two-way, "copy" is (src, dst).
PHASES_P1P2 = [
    ("P1", "swap", ("A", "D")),      # swap A <-> D
    ("P2", "copy", ("E", "B")),      # B <- E
]
# Sheet 1 also shows a third phase; keep it available but off by default.
PHASES_ALL3 = PHASES_P1P2 + [("P3", "swap", ("C", "E"))]


def build_stream(phases, reps=25, n_steps=10, seed=42):
    """One persistent table, mutated cumulatively, snapshotted after every step.
    Each phase draws its own without-replacement schedule, so a phase touches
    every row exactly once across its 10 steps."""
    df = make_base_block(reps)
    snaps = [("t00", "baseline", df.copy())]
    rng = np.random.default_rng(seed)

    for name, kind, cols in phases:
        sched = make_schedule(len(df), n_steps, rng)
        for i, rows in enumerate(sched, start=1):
            if kind == "swap":
                swap_cells(df, rows, *cols)
            elif kind == "copy":
                copy_cells(df, rows, *cols)
            else:
                raise ValueError(f"unknown phase kind: {kind}")
            snaps.append((f"{name}.{i:02d}", name, df.copy()))
    return snaps


def necessary_features(df):
    """A feature is NECESSARY if dropping it makes Y unpredictable from
    everything else -- i.e. it is irreplaceable across the Rashomon set.
    This is the property MCR's lower bound is designed to detect."""
    return [f for f in FEATURES
            if not determines(df, [x for x in FEATURES if x != f])]


def reliance_report(snaps):
    out = []
    for label, phase, d in snaps:
        pairs = [p for p in combinations(FEATURES, 2) if determines(d, p)]
        out.append({
            "t": label,
            "phase": phase,
            "solvable": determines(d, FEATURES),        # is 100% accuracy attainable?
            "n_pairs": len(pairs),                     # size of the Rashomon set
            "relevant": "".join(sorted({f for p in pairs for f in p})),
            "necessary": "".join(necessary_features(d)) or "-",
            "singleton": "".join(f for f in FEATURES if determines(d, [f])) or "-",
            "sufficient pairs": " ".join("".join(p) for p in pairs),
        })
    return pd.DataFrame(out)


snaps = build_stream(PHASES_P1P2, seed=42)
rep   = reliance_report(snaps)

# THE validation assertions -- these encode "performance will never change"
assert len(snaps) == 21
assert rep["solvable"].all(), "a perfect predictor must exist at EVERY timestep"
assert (rep["n_pairs"] >= 1).all(), "at least one 2-feature predictor must survive"
assert (rep["singleton"] == "-").all(), "no single feature may ever suffice"

pd.set_option("display.width", 200)
print(rep.to_string(index=False))

# tidy long frame for the modelling cells
stream = pd.concat([d.assign(t=l, phase=p) for l, p, d in snaps], ignore_index=True)
print("\nstream:", stream.shape)

    t    phase  solvable  n_pairs relevant necessary singleton  sufficient pairs
  t00 baseline      True        6    ABCDE         -         - AC AD AE BC BD BE
P1.01       P1      True        3    ABCDE         -         -          AD BC BE
P1.02       P1      True        3    ABCDE         -         -          AD BC BE
P1.03       P1      True        3    ABCDE         -         -          AD BC BE
P1.04       P1      True        3    ABCDE         -         -          AD BC BE
P1.05       P1      True        3    ABCDE         -         -          AD BC BE
P1.06       P1      True        3    ABCDE         -         -          AD BC BE
P1.07       P1      True        3    ABCDE         -         -          AD BC BE
P1.08       P1      True        3    ABCDE         -         -          AD BC BE
P1.09       P1      True        3    ABCDE         -         -          AD BC BE
P1.10       P1      True        6    ABCDE         -         - AB AD BC BE CD DE
P2.01       P2      True    

In [4]:
from sklearn.ensemble import RandomForestClassifier

RF_KW = dict(n_estimators=300, max_features="sqrt", bootstrap=True, n_jobs=-1)

def fit_forest(df, seed=0):
    clf = RandomForestClassifier(random_state=seed, **RF_KW)
    clf.fit(df[FEATURES], df["Y"])
    return clf


def per_tree_votes(clf, X):
    """(n_trees, n_rows) array of each individual tree's predicted label.
    Note sklearn's RF itself averages probabilities (soft voting), so these
    hard votes are a deliberately separate statistic."""
    X = np.asarray(X, dtype=np.float32)
    return np.stack([clf.classes_[t.predict(X).astype(int)] for t in clf.estimators_])


def vote_dominance(clf, X):
    """Per row: share of trees voting with the majority.
    1.0 = unanimous, 0.5 = the forest is split down the middle.
    Requires NO labels -- this is the point."""
    p1 = per_tree_votes(clf, X).mean(axis=0)
    return np.maximum(p1, 1.0 - p1)


def dominance_summary(dom):
    """Report the distribution, not just the mean -- the mean is a proxy
    for accuracy, the tails are the actual signal."""
    return {"dom_mean": dom.mean(),
            "dom_min": dom.min(),
            "frac_not_unanimous": float((dom < 1.0).mean())}


def usage_profile(clf):
    """Which features the trees actually split on -- 'usage dominance'."""
    n_nodes     = np.zeros(len(FEATURES))
    trees_using = np.zeros(len(FEATURES))
    for t in clf.estimators_:
        f = t.tree_.feature
        used = f[f >= 0]                      # -2 marks leaf nodes
        n_nodes     += np.bincount(used, minlength=len(FEATURES))
        trees_using += np.bincount(np.unique(used), minlength=len(FEATURES))
    return pd.DataFrame({
        "feature": FEATURES,
        "mdi": clf.feature_importances_,
        "split_share": n_nodes / n_nodes.sum(),
        "trees_using": trees_using / len(clf.estimators_),
    })


# --- sanity check at t00: the forest must be perfect AND unanimous ---
t0     = snaps[0][2]
frozen = fit_forest(t0, seed=0)

hard = (per_tree_votes(frozen, t0[FEATURES]).mean(axis=0) > 0.5).astype(int)
assert frozen.score(t0[FEATURES], t0["Y"]) == 1.0
assert (hard == frozen.predict(t0[FEATURES])).all()

dom0 = vote_dominance(frozen, t0[FEATURES])
assert dom0.min() == 1.0, "at t00 every tree must agree on every row"

print("t00 accuracy:", frozen.score(t0[FEATURES], t0["Y"]))
print("t00 dominance:", dominance_summary(dom0))
print()
print(usage_profile(frozen).round(3).to_string(index=False))

t00 accuracy: 1.0
t00 dominance: {'dom_mean': np.float64(1.0), 'dom_min': np.float64(1.0), 'frac_not_unanimous': 0.0}

feature   mdi  split_share  trees_using
      A 0.289        0.261        0.643
      B 0.318        0.276        0.660
      C 0.127        0.152        0.427
      D 0.138        0.149        0.397
      E 0.128        0.162        0.443


In [5]:
frozen = fit_forest(snaps[0][2], seed=0)   # deployed at t00, never updated

rows = []
for label, phase, d in snaps:
    X, y = d[FEATURES], d["Y"]
    fd   = vote_dominance(frozen, X)       # label-free, on the frozen model
    clf  = fit_forest(d, seed=0)           # retrained on the current concept
    mdi  = usage_profile(clf).set_index("feature")["mdi"]

    rec = {"t": label, "phase": phase,
           "froz_acc": frozen.score(X, y),
           "dom_mean": fd.mean(), "dom_min": fd.min(),
           "frac_ne": float((fd < 1.0).mean()),
           "retr_acc": clf.score(X, y)}
    rec.update({f"mdi_{f}": mdi[f] for f in FEATURES})
    rec["top"] = mdi.idxmax()
    rows.append(rec)

res = pd.DataFrame(rows).merge(rep[["t", "n_pairs", "necessary"]], on="t")

assert (res["retr_acc"] == 1.0).all(), "retrained arm must never lose accuracy"
assert res.loc[0, "top"] in ("A", "B")
assert res.iloc[-1]["top"] == "D"

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 50)
print(res.round(3).to_string(index=False))
print("\nD overtakes A at:", res.loc[res.index[res.mdi_D > res.mdi_A][0], "t"],
      "| B falls below A at:", res.loc[res.index[res.mdi_B < res.mdi_A][0], "t"])

    t    phase  froz_acc  dom_mean  dom_min  frac_ne  retr_acc  mdi_A  mdi_B  mdi_C  mdi_D  mdi_E top  n_pairs necessary
  t00 baseline      1.00     1.000    1.000     0.00       1.0  0.289  0.318  0.127  0.138  0.128   B        6         -
P1.01       P1      0.98     0.976    0.513     0.05       1.0  0.304  0.328  0.131  0.128  0.109   B        3         -
P1.02       P1      0.94     0.952    0.513     0.10       1.0  0.273  0.314  0.128  0.171  0.114   B        3         -
P1.03       P1      0.92     0.929    0.513     0.15       1.0  0.247  0.338  0.128  0.187  0.100   B        3         -
P1.04       P1      0.89     0.891    0.513     0.23       1.0  0.237  0.330  0.124  0.214  0.095   B        3         -
P1.05       P1      0.86     0.877    0.513     0.26       1.0  0.218  0.325  0.118  0.246  0.094   B        3         -
P1.06       P1      0.84     0.867    0.513     0.28       1.0  0.214  0.315  0.128  0.251  0.092   B        3         -
P1.07       P1      0.82     0.8

In [6]:
import warnings, shap
warnings.simplefilter("ignore")

def mean_abs_shap(shap_values, n_features):
    """Same helper as your rotatingHyperplane notebook, so the two sets of
    numbers are directly comparable."""
    sv = np.abs(np.array(shap_values))
    feat_axis = [ax for ax, size in enumerate(sv.shape) if size == n_features][-1]
    other = tuple(ax for ax in range(sv.ndim) if ax != feat_axis)
    return sv.mean(axis=other)

def shap_share(clf, X):
    imp = mean_abs_shap(shap.TreeExplainer(clf).shap_values(X), X.shape[1])
    return imp / imp.sum()

rows = []
for label, phase, d in snaps:
    X = d[FEATURES]
    r = shap_share(fit_forest(d, seed=0), X)   # retrained arm
    f = shap_share(frozen, X)                  # frozen arm, same t00 model
    rows.append({"t": label,
                 **{f"retr_{k}": v for k, v in zip(FEATURES, r)},
                 **{f"froz_{k}": v for k, v in zip(FEATURES, f)}})

sh = pd.DataFrame(rows)
print(sh.round(3).to_string(index=False))

R = sh[[f"retr_{f}" for f in FEATURES]].to_numpy()
F = sh[[f"froz_{f}" for f in FEATURES]].to_numpy()
print("\nSHAP swing (max-min) per feature")
print("  retrained:", np.round(R.max(0) - R.min(0), 3), "| largest:", round((R.max(0)-R.min(0)).max(), 3))
print("  frozen:   ", np.round(F.max(0) - F.min(0), 3), "| largest:", round((F.max(0)-F.min(0)).max(), 3))
print("\ntruth says D becomes necessary; frozen SHAP puts D at:",
      f"{F[0,3]:.3f} -> {F[-1,3]:.3f}",
      "(direction:", "WRONG)" if F[-1,3] < F[0,3] else "right)")

# --- what kind of drift is this, formally? ---
allrows = pd.concat([d for l, p, d in snaps], ignore_index=True)
g = allrows.groupby(FEATURES, observed=True)["Y"].nunique()
marg = pd.DataFrame({l: d[FEATURES].mean() for l, p, d in snaps}).T
print("\ndistinct feature vectors over whole stream:", len(g))
print("any vector with conflicting Y  ->  true P(Y|X) drift?", bool((g > 1).any()))
print("marginal P(x=1) range:", {c: f"{marg[c].min():.2f}-{marg[c].max():.2f}" for c in FEATURES})
print("support size t00 / mid-P1 / end-P1:",
      *[len(set(map(tuple, snaps[i][2][FEATURES].to_numpy()))) for i in (0, 5, 10)])

/Users/shreyu/VSCODE/junk/UoN-Business-Analytics/Dissertation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


    t  retr_A  retr_B  retr_C  retr_D  retr_E  froz_A  froz_B  froz_C  froz_D  froz_E
  t00   0.246   0.254   0.167   0.154   0.179   0.246   0.254   0.167   0.154   0.179
P1.01   0.243   0.264   0.164   0.167   0.163   0.247   0.256   0.166   0.153   0.178
P1.02   0.246   0.256   0.150   0.187   0.162   0.249   0.259   0.165   0.151   0.176
P1.03   0.224   0.269   0.158   0.189   0.160   0.250   0.261   0.165   0.150   0.175
P1.04   0.221   0.261   0.155   0.211   0.152   0.253   0.265   0.163   0.147   0.172
P1.05   0.226   0.250   0.153   0.227   0.144   0.254   0.267   0.163   0.146   0.171
P1.06   0.226   0.251   0.155   0.228   0.141   0.255   0.268   0.163   0.145   0.169
P1.07   0.229   0.243   0.146   0.242   0.140   0.256   0.270   0.162   0.144   0.168
P1.08   0.183   0.266   0.173   0.232   0.145   0.259   0.275   0.159   0.141   0.166
P1.09   0.174   0.267   0.167   0.239   0.153   0.262   0.280   0.157   0.138   0.163
P1.10   0.167   0.243   0.158   0.257   0.175   0.266 

In [7]:
# NOTE: river wheels for Python 3.14 may not exist. If this import fails,
# that is the point at which the 3.11/3.12 pin stops being optional.
from river.drift import ADWIN
from river.drift.binary import DDM     # river >= 0.21; older: river.drift.DDM

N = len(snaps[0][2])          # 100 instances per window

# per-instance error streams, in stream order
err_frozen, err_retrained = [], []
for label, phase, d in snaps:
    X, y = d[FEATURES], d["Y"].to_numpy()
    err_frozen.extend((frozen.predict(X) != y).astype(int).tolist())
    clf = fit_forest(d, seed=0)
    err_retrained.extend((clf.predict(X) != y).astype(int).tolist())


def first_alarm(errors, detector):
    for i, e in enumerate(errors):
        detector.update(e)
        if detector.drift_detected:
            return i
    return None


for name, errs in [("frozen", err_frozen), ("retrained", err_retrained)]:
    print(f"{name:<10} errors: {sum(errs):>5} / {len(errs)}")
    for dname, det in [("ADWIN", ADWIN()), ("DDM", DDM())]:
        a = first_alarm(errs, det)
        where = "never" if a is None else f"instance {a}  (window {a//N} = {snaps[a//N][0]})"
        print(f"   {dname:<6} first alarm: {where}")

frozen     errors:   521 / 2100
   ADWIN  first alarm: instance 863  (window 8 = P1.08)
   DDM    first alarm: instance 121  (window 1 = P1.01)
retrained  errors:     0 / 2100
   ADWIN  first alarm: never
   DDM    first alarm: never


In [8]:
SEEDS = range(8)

def window_signals(d):
    """Refit under several seeds so real movement can be separated from
    tie-breaking noise between the duplicated columns."""
    X = d[FEATURES]
    mdi, shp = [], []
    for s in SEEDS:
        c = fit_forest(d, seed=s)
        mdi.append(c.feature_importances_)
        shp.append(shap_share(c, X))
    return np.array(mdi), np.array(shp), shap_share(frozen, X), vote_dominance(frozen, X)


sig = [window_signals(d) for l, p, d in snaps]

# reference = mean signal at t00; null band = how far same-concept refits wander from it
mdi_ref, shp_ref = sig[0][0].mean(0), sig[0][1].mean(0)
null_mdi = np.abs(sig[0][0] - mdi_ref).sum(1).max()
null_shp = np.abs(sig[0][1] - shp_ref).sum(1).max()

dep = pd.DataFrame([{
    "t": snaps[i][0],
    "mdi_L1":      np.abs(m.mean(0) - mdi_ref).sum(),
    "shap_L1":     np.abs(s.mean(0) - shp_ref).sum(),
    "frozShap_L1": np.abs(fz - shp_ref).sum(),
    "dom_min":     dm.min(),
    "frac_ne":     float((dm < 1.0).mean()),
} for i, (m, s, fz, dm) in enumerate(sig)])

print(f"null L1 band from {len(list(SEEDS))} refits on identical t00 data: "
      f"MDI {null_mdi:.4f}   SHAP {null_shp:.4f}")
print(f"MDI departure at P1.01 is {dep.loc[1,'mdi_L1']:.4f} -- "
      f"{'INSIDE' if dep.loc[1,'mdi_L1'] < null_mdi else 'outside'} the noise band\n")
print(dep.round(3).to_string(index=False))

def first(col, thr):
    idx = np.where(dep[col].to_numpy() > thr)[0]
    return dep.loc[idx[0], "t"] if len(idx) else "never"

print("\n%-28s %-9s %-11s %s" % ("signal", "labels?", "null band", "first alarm"))
for nm, lab, thr, col in [("vote dominance (frozen)", "no",  "0.0000",          "frac_ne"),
                          ("retrained MDI L1",        "yes", f"{null_mdi:.4f}", "mdi_L1"),
                          ("retrained SHAP L1",       "yes", f"{null_shp:.4f}", "shap_L1"),
                          ("frozen SHAP L1",          "no",  f"{null_shp:.4f}", "frozShap_L1")]:
    print("%-28s %-9s %-11s %s" % (nm, lab, thr, first(col, float(thr))))

null L1 band from 8 refits on identical t00 data: MDI 0.1027   SHAP 0.0712
MDI departure at P1.01 is 0.0590 -- INSIDE the noise band

    t  mdi_L1  shap_L1  frozShap_L1  dom_min  frac_ne
  t00   0.000    0.000        0.028    1.000     0.00
P1.01   0.059    0.063        0.030    0.513     0.05
P1.02   0.104    0.088        0.031    0.513     0.10
P1.03   0.148    0.109        0.033    0.513     0.15
P1.04   0.202    0.133        0.039    0.513     0.23
P1.05   0.220    0.144        0.042    0.513     0.26
P1.06   0.221    0.150        0.047    0.513     0.28
P1.07   0.214    0.159        0.053    0.513     0.31
P1.08   0.277    0.179        0.067    0.513     0.38
P1.09   0.293    0.193        0.083    0.513     0.44
P1.10   0.367    0.193        0.102    0.513     0.50
P2.01   0.539    0.344        0.134    0.507     0.54
P2.02   0.660    0.399        0.166    0.507     0.58
P2.03   0.807    0.464        0.222    0.507     0.64
P2.04   0.925    0.523        0.294    0.507     0.71
P2